# StravaGANte Syntetic Data Retriever
The dataset is collected using OpenRouteService, an open source project which exposes free APIs.

In [37]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv
from duckduckgo_search import DDGS
import csv
import sys
from time import sleep

profiles = [
    'driving-car',
    'driving-hgv',
    'cycling-regular',
    'cycling-road',
    'cycling-mountain',
    'cycling-electric'
]
output_filedir = 'Data/Syntetic/'
output_filename = 'output.gpx'
os.makedirs(output_filedir, exist_ok=True)

load_dotenv()
ors_token = os.getenv("OpenRouteServiceApiKey")
print("Token ok.") if ors_token else print("Token not found.")

Token ok.


IMDB 250 top films to scrape locations

In [44]:
def print_progress_bar(iteration, total, length=50):
    percent = ("{0:.1f}").format(100 * (iteration / float(total)))
    filled_length = int(length * iteration // total)
    bar = '█' * filled_length + '-' * (length - filled_length)
    sys.stdout.write(f'\rProgress: |{bar}| {percent}%\n')
    sys.stdout.flush()

latlong_file = '../Data/latlong_movies.csv'
df_movies = pd.read_csv('../Data/IMDB_Top_250_Movies.csv', usecols=[1])
movie_names = df_movies['name'].tolist()
total_rows = len(movie_names)

movie_locations = {}

if os.path.isfile(latlong_file):
    existing_data = pd.read_csv(latlong_file)
    existing_data.dropna(subset=['name', 'latlong_url'], inplace=True)
    existing_data.to_csv(latlong_file, index=False)
    existing_movies = existing_data['name'].tolist()
    completed_rows = existing_data['latlong_url'].notna().sum()
else:
    existing_movies = []
    with open(latlong_file, mode='a', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(['name', 'latlong_url'])

for movie in movie_names:
    if movie not in existing_movies:
        print(f"{movie}...")
        query = f'site:latlong.net/location/ {movie}'
        results = DDGS().text(f'site:latlong.net/location/ {movie}', max_results=1)
        sleep(10)
        href = results[0]['href'] if results else None
        if href:
            with open(latlong_file, mode='a', newline='') as file:
                writer = csv.writer(file)
                writer.writerow([movie, href])
            print(f"Found.")
            completed_rows+=1
        else:
            print(f"No results found.")
        
    print_progress_bar(completed_rows, total_rows)

Progress: |███████-------------------------------------------| 14.0%
Progress: |███████-------------------------------------------| 14.0%
Progress: |███████-------------------------------------------| 14.0%
Progress: |███████-------------------------------------------| 14.0%
Progress: |███████-------------------------------------------| 14.0%
Progress: |███████-------------------------------------------| 14.0%
Progress: |███████-------------------------------------------| 14.0%
Progress: |███████-------------------------------------------| 14.0%
Progress: |███████-------------------------------------------| 14.0%


Progress: |███████-------------------------------------------| 14.0%
Progress: |███████-------------------------------------------| 14.0%
Progress: |███████-------------------------------------------| 14.0%
Progress: |███████-------------------------------------------| 14.0%
Progress: |███████-------------------------------------------| 14.0%
Progress: |███████-------------------------------------------| 14.0%
Progress: |███████-------------------------------------------| 14.0%
Progress: |███████-------------------------------------------| 14.0%
Progress: |███████-------------------------------------------| 14.0%
Progress: |███████-------------------------------------------| 14.0%
Progress: |███████-------------------------------------------| 14.0%
Progress: |███████-------------------------------------------| 14.0%
Progress: |███████-------------------------------------------| 14.0%
Progress: |███████-------------------------------------------| 14.0%
Progress: |███████----------------

RatelimitException: https://duckduckgo.com/?q=site%3Alatlong.net%2Flocation%2F+Whiplash 202 Ratelimit

Scrape from latlong.net every location.

In [7]:
import requests
from bs4 import BeautifulSoup

latlong_movies_copy_file = '../Data/latlong_movies_copy.csv'
output_file = '../Data/latlong_movies_coordinates.csv'

# Read the CSV file
df_latlong = pd.read_csv(latlong_movies_copy_file)

# Create a list to store the results
results = []

# Iterate over each row in the DataFrame
for index, row in df_latlong.iterrows():
    url = row['latlong_url']
    movie_name = row['name']
    
    # Make a request to the URL
    response = requests.get(url)
    
    if response.status_code == 200:
        # Parse the HTML content
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Find the table in the page
        table = soup.find('table')
        
        if table:
            rows = table.find_all('tr')
            for r in rows[1:]:
                lat = r.find_all('td')[1].text.strip()
                lon = r.find_all('td')[2].text.strip()
                location_name = r.find_all('td')[0].text.strip()
            
                # Append the result to the list
                results.append([movie_name, location_name, lat, lon])
        else:
            print(f"No table found for {movie_name}")
    else:
        print(f"Failed to retrieve {url}")

# Create a DataFrame from the results
df_results = pd.DataFrame(results, columns=['name', 'location_name', 'latitude', 'longitude'])

# Save the results to a new CSV file
df_results.to_csv(output_file, index=False)

POST Request to OpenRouteService

In [ ]:
pz = [11.88713795374955,45.41111616690249]
body = {"coordinates":[pz,[11.931983982915773, 45.42370290352176], pz]} # torre archimede and prato della valle

headers = {
    'Accept': 'application/json, application/geo+json, application/gpx+xml, img/png; charset=utf-8',
    'Authorization': ors_token,
    'Content-Type': 'application/json; charset=utf-8'
}
call = requests.post(f'https://api.openrouteservice.org/v2/directions/{profiles[2]}/gpx', json=body, headers=headers)

if call.status_code == 200:
    gpx_file_path = os.path.join(output_filedir, output_filename)
    with open(gpx_file_path, 'w') as file:
        file.write(call.text)